# 04 — Feature Selection Scenarios

Four **input scenarios** are compared for segmentation. No multi-stage
selection — each scenario is a single, direct definition of the input channels:

| key | scenario | channels |
|-----|----------|----------|
| `single_date` | Single date (peak-NDVI, ~14 July), all bands, **no selection** | 10 |
| `mt_base` | Multi-temporal NDVI — 4 dates (per-quarter peak NDVI) × bands | 4×bands |
| `gsi` | **GSI** feature selection, per-crop **normalized score ≥ 0.5** | variable |
| `rf`  | **RF importance ranking**, per-crop **normalized score ≥ 0.5** | variable |

GSI and RF both score every (date × band) channel, **min-max normalize the score
per crop to [0,1]**, keep channels with score ≥ threshold, then union across crops
(Wei et al. 2023). This notebook runs both at threshold 0.5 and compares the sets.

> Heavy: reads the full processed S2 stack. Needs `data/processed/` locally.

In [ ]:
# Register this repo as `crop_mapping_pipeline` regardless of checkout dir name,
# and silence MLflow telemetry.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'crop_mapping_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from crop_mapping_pipeline import config as C
print('Repo   :', REPO)
print('Classes:', C.NUM_CLASSES, '| crops:', list(C.CDL_CLASS_NAMES.values()))
print('Norms  :', C.__dict__.get('NORM_MODES', ('percentile', 'minmax', 'zscore')))

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from crop_mapping_pipeline.stages.selection import band_scoring
from crop_mapping_pipeline.stages.selection.gsi_selection import run_gsi_direct
from crop_mapping_pipeline.stages.selection.feature_importance_selection import run_rf_direct

THRESH = 0.5   # normalized-score threshold (main setting)
OUT_BASE = C.PROCESSED_DIR

## 1. GSI selection (normalized score ≥ 0.5)

Per-crop Global Separability Index over all channels → min-max normalize → keep ≥ 0.5 → union.
Writes `select_gsi_direct_s0.5.json` (the file the training reads for `--exp gsi`).

In [ ]:
years_data = [band_scoring.get_train_year_inputs()]   # [(year, s2_files, cdl_path)] — v6.1 = 2024
run_gsi_direct(
    years_data,
    score_threshold=THRESH,
    data_dir=str(OUT_BASE),
    out_stem=f'select_gsi_direct_s{THRESH:g}',
)

## 2. RF importance selection (normalized score ≥ 0.5)

Multi-class Random Forest (per-class MDI) over all channels → min-max normalize → keep ≥ 0.5 → union.
Writes `select_rf_direct_s0.5.json`.

In [ ]:
run_rf_direct(
    years_data,
    score_threshold=THRESH,
    data_dir=str(OUT_BASE),
    out_stem=f'select_rf_direct_s{THRESH:g}',
)

## 3. Compare the selected channel sets

Expectation: GSI keeps a broad, year-spread set; RF concentrates on fewer channels.

In [ ]:
def load_union(stem):
    p = OUT_BASE / f'{stem}.json'
    if not p.exists():
        print('missing:', p); return set()
    d = json.loads(p.read_text())
    chans = set()
    for v in (d.values() if isinstance(d, dict) else []):
        if isinstance(v, dict):
            for key in ('channels', 'bands', 'selected', 'indices'):
                if key in v: chans.update(map(str, v[key]))
        elif isinstance(v, list):
            chans.update(map(str, v))
    return chans

gsi = load_union(f'select_gsi_direct_s{THRESH:g}')
rf  = load_union(f'select_rf_direct_s{THRESH:g}')
if gsi and rf:
    print(f'GSI : {len(gsi)} channels')
    print(f'RF  : {len(rf)} channels')
    print(f'Shared {len(gsi & rf)} | GSI-only {len(gsi - rf)} | RF-only {len(rf - gsi)}')
    plt.figure(figsize=(5, 4))
    plt.bar(['GSI', 'RF'], [len(gsi), len(rf)], color=['steelblue', 'darkorange'])
    plt.ylabel('# channels (score ≥ 0.5)')
    plt.title('Selected channel count'); plt.tight_layout(); plt.show()